# Cocopila: Pandas Data Agent Pipeline — Kaggle Execution Notebook 🥥📊⚡

Notebook này nạp mô hình **Qwen3.5-4B** trên 2 GPU Tesla T4 song song (`tensor_parallel_size=2`) bằng **vLLM Python API Native** và thực thi toàn bộ **LangGraph Agent Pipeline** (5 Nodes + Reflection Loop).

In [ ]:
# 1. Cài đặt các gói phụ thuộc dự án kèm Constraint Pinning (Tránh xung đột môi trường Kaggle)
print("📥 Đang cài đặt dependencies...")
!pip install -q \
    "protobuf>=5.26.1,<6.0dev" \
    "starlette>=0.40.0,<1.0" \
    "opentelemetry-api>=1.35.0,<1.39.0" \
    "opentelemetry-sdk>=1.35.0,<1.39.0" \
    "numba>=0.60.0,<0.63.0" \
    "google-cloud-bigquery-storage>=2.30.0,<3.0.0" \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    langchain-openai>=0.2.0 \
    vllm>=0.6.0 \
    pyyaml>=6.0 \
    json-repair>=0.30.0 \
    openpyxl>=3.1.0 \
    tabulate>=0.9.0 \
    thefuzz>=0.22.0

print("✅ Cài đặt phụ thuộc thành công!")

In [ ]:
import os
import sys
from kaggle_secrets import UserSecretsClient

# ==========================================
# 1. CẤU HÌNH MÔI TRƯỜNG KAGGLE DUAL GPU
# ==========================================
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["OMP_NUM_THREADS"] = "4"

# ==========================================
# 2. TỰ ĐỘNG QUÉT MODEL TỪ KAGGLE INPUT
# ==========================================
def get_model_path(default_hf_id="Qwen/Qwen3.5-4B"):
    input_dir = "/kaggle/input"
    if os.path.exists(input_dir):
        for root, dirs, files in os.walk(input_dir):
            if "config.json" in files or any(f.endswith(".safetensors") for f in files):
                print(f"📁 Đã tìm thấy local model tại Kaggle Input: {root}")
                return root
    return default_hf_id

MODEL_ID = get_model_path("Qwen/Qwen3.5-4B")

# ==========================================
# 3. KHỞI TẠO VLLM PYTHON NATIVE (DUAL GPU)
# ==========================================
from vllm import LLM, SamplingParams

print(f"🚀 Khởi tạo vLLM Native Python Engine trên 2 GPU T4 (Model: {MODEL_ID})...")

llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=2,        # 🟢 Chia đều model cho 2 GPU T4
    dtype="float16",               # T4 float16
    gpu_memory_utilization=0.75,   # Giữ VRAM chừa bộ nhớ PyTorch IPC
    max_model_len=2048,
    enforce_eager=True,
    disable_custom_all_reduce=True,
    trust_remote_code=True
)

print("✅ Khởi tạo vLLM Native Engine 2 GPU THÀNH CÔNG!")

In [ ]:
# 3. TẠO OPENAI-COMPATIBLE API SERVER TRONG BACKGROUND THREAD (CHO LANGCHAIN)
import threading
from fastapi import FastAPI
import uvicorn
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI()

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.0
    max_tokens: Optional[int] = 512

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    prompt = "\n".join([f"{m.role}: {m.content}" for m in req.messages])
    sp = SamplingParams(temperature=req.temperature, max_tokens=req.max_tokens)
    res = llm.generate([prompt], sp)
    text = res[0].outputs[0].text
    return {
        "id": "chatcmpl-cocopila",
        "object": "chat.completion",
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": text},
            "finish_reason": "stop"
        }]
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🌐 OpenAI API Server sẵn sàng tại http://localhost:8000/v1 !")

In [ ]:
# 4. THỰC THI TOÀN BỘ COCOPILA LANGGRAPH PIPELINE (STEP 6 & 7)
import sys
sys.path.insert(0, "/kaggle/working/cocopila")

from pipeline.src.graph import create_cocopila_graph

# Khởi tạo LangGraph Agent Graph
agent = create_cocopila_graph()

# Truy vấn mẫu
user_query = "Tính tổng doanh thu và số lượng đơn hàng theo dòng sản phẩm từ file sample_sales"

initial_state = {
    "user_query": user_query,
    "status": "pending",
    "retry_count": 0,
    "node_latencies": {}
}

print(f"🚀 Bắt đầu chạy Agent Pipeline cho câu hỏi: '{user_query}'...")
final_state = agent.invoke(initial_state)

print("\n=================== KẾT QUẢ XỬ LÝ AGENT ===================")
print("📌 Trạng thái (Status):", final_state.get("status"))
print("📌 Phân tích Intent (Parsed Query):", final_state.get("parsed_query"))
print("📌 File dữ liệu khớp (Matched File):", final_state.get("matched_table_path"))
print("📌 Ánh xạ Cột (Column Mapping):", final_state.get("column_mapping"))
print("📌 Mã Pandas Sinh Ra (Generated Code):\n", final_state.get("generated_code"))
print("📌 Kết Quả Thực Thi (Execution Result):\n", final_state.get("execution_result"))
print("📌 Thời gian xử lý từng Node (Latencies):", final_state.get("node_latencies"))
print("===========================================================")